In [1]:
# https://judge.nitro-ai.org/competitions/roai-2025/baraj-nationala-2026/2/view

import os 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
from tqdm.auto import tqdm 

In [2]:
data = pd.read_csv("test_data.csv")
data.shape

(7000, 2)

In [3]:
data

,datapointID,text
0,0,Cleric in Najaf Refuses to Meet Iraqi Mediator...
1,1,School #39;s out to shun IE Citing security ri...
2,2,How do I begin to review a film that will soon...
3,3,ANALYSIS-Suspected Islamist killing tests Dutc...
4,4,Venezuela Boosts Taxes on Orinoco Deals &lt;p&...
...,...,...
6995,6995,35 accused of Qaeda-linked plot to bomb target...
6996,6996,Afghan President Aborts Trip After Rocket Atta...
6997,6997,European Shares at 2-Month High European share...
6998,6998,Oracle warns of exploits for latest DB flaws O...


In [4]:
from transformers import pipeline

pipe = pipeline("feature-extraction", model="BAAI/bge-small-en-v1.5")

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [23]:
def get_embeddings(texts):
    embeddings = []
    for i in tqdm(range(len(texts))):
        emb = np.array(pipe(data['text'][i])).squeeze()[0]
        embeddings.append(emb)
    embeddings = np.stack(embeddings)
    return embeddings

embeddings = get_embeddings(data['text'].tolist())

  0%|          | 0/7000 [00:00<?, ?it/s]

In [25]:
embeddings.shape

(7000, 384)

In [ ]:
from sklearn.cluster import KMeans 

clusterer = KMeans(n_clusters=2, n_init=10, random_state=43)

labels1 = clusterer.fit_predict(embeddings)
np.bincount(labels1)

array([6177,  823])

In [43]:
guess_num_clusters = 4

clusterer = KMeans(n_clusters=guess_num_clusters, n_init=10, random_state=42)

labels2 = clusterer.fit_predict(embeddings)
labels2[np.where(labels1 == 1)] = -1
labels2

array([ 0,  3, -1, ...,  3,  3,  3], shape=(7000,), dtype=int32)

In [44]:
datapoint_ids = data['datapointID'] 

# Subtask 1
answer_subtask1 = labels1
subtask1_df = pd.DataFrame({
    'subtaskID': 1,
    'datapointID': datapoint_ids,
    'answer': answer_subtask1
})

# Subtask 2
answer_subtask2 = labels2

subtask2_df = pd.DataFrame({
    'subtaskID': 2,
    'datapointID': datapoint_ids,
    'answer': answer_subtask2
})

submission_df = pd.concat([subtask1_df, subtask2_df], ignore_index=True)

submission_file = "submission.csv"
submission_df.to_csv(submission_file, index=False)

submission_df[submission_df['subtaskID']==2]['answer'].value_counts()

answer
 3    2815
 0    2319
 2    1037
-1     823
 1       6
Name: count, dtype: int64